# H4: PDE-Constrained Loss Functions Suppress Gradient Vanishing

**Hypothesis**: Embedding PDE constraints into VQC loss functions yields gradient variance that decays polynomially rather than exponentially with qubit count.

**References**: Ukwatta Hewage et al. (2026), arXiv:2311.04965

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import json
import os
from datetime import datetime

np.random.seed(42)
qml.numpy.random.seed(42)

In [ ]:
QUBIT_RANGE = [2, 3, 4, 5, 6]
N_SAMPLES = 80
ALPHA = 0.1
EPS = 0.01

def global_circuit(params, n_qubits):
    for i in range(n_qubits):
        qml.RY(np.pi / 4, wires=i)
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])
    H = np.zeros((2 ** n_qubits, 2 ** n_qubits))
    H[0, 0] = 1
    return qml.expval(qml.Hermitian(H, list(range(n_qubits))))

def local_circuit(params, n_qubits):
    for i in range(n_qubits):
        qml.RY(np.pi / 4, wires=i)
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

def pde_constrained_circuit(params, n_qubits):
    for i in range(n_qubits):
        qml.RY(np.pi / 4, wires=i)
    for i in range(n_qubits):
        qml.RX(params[i], wires=i)
    for i in range(n_qubits - 1):
        qml.CZ(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

In [ ]:
def compute_grad_var(circuit_fn, n_qubits):
    dev = qml.device("default.qubit", wires=n_qubits)
    qnode = qml.QNode(circuit_fn, dev, diff_method="parameter-shift")
    grad_fn = qml.grad(qnode)
    grads = []
    for _ in range(N_SAMPLES):
        p = np.random.uniform(-np.pi, np.pi, size=n_qubits)
        g = grad_fn(p, n_qubits=n_qubits)
        grads.append(g[-1])
    return float(np.var(np.array(grads)))

def pde_cost(params, n_qubits):
    dev = qml.device("default.qubit", wires=n_qubits)
    qnode = qml.QNode(pde_constrained_circuit, dev, diff_method="parameter-shift")
    base_val = qnode(params, n_qubits=n_qubits)
    residual = 0.0
    for i in range(len(params)):
        pp = params.copy(); pp[i] += EPS
        residual += (qnode(pp, n_qubits=n_qubits) - base_val) ** 2
    return base_val + ALPHA * residual / len(params)

def compute_pde_grad_var(n_qubits):
    grads = []
    for _ in range(N_SAMPLES):
        p = np.random.uniform(-np.pi, np.pi, size=n_qubits)
        g = []
        for i in range(n_qubits):
            pp = p.copy(); pp[i] += EPS
            pm = p.copy(); pm[i] -= EPS
            g.append((pde_cost(pp, n_qubits) - pde_cost(pm, n_qubits)) / (2 * EPS))
        grads.append(g[-1])
    return float(np.var(np.array(grads)))

In [ ]:
results = {}
for name, fn in [("Global cost", global_circuit), ("Local cost", local_circuit)]:
    vars_list = []
    for n in QUBIT_RANGE:
        v = compute_grad_var(fn, n)
        vars_list.append(v)
        print(f"  {name:>15} n={n}: var={v:.6e}")
    results[name] = vars_list

pde_vars = []
for n in QUBIT_RANGE:
    v = compute_pde_grad_var(n)
    pde_vars.append(v)
    print(f"  PDE-constrained n={n}: var={v:.6e}")
results["PDE-constrained"] = pde_vars

os.makedirs('results/H4', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'results/H4/run_{ts}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to results/H4/run_{ts}.json")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
styles = {"Global cost": "o-", "Local cost": "s--", "PDE-constrained": "D-."}
for name, vars_list in results.items():
    ax.semilogy(QUBIT_RANGE, vars_list, styles[name], label=name, linewidth=2, markersize=8)

n_arr = np.array(QUBIT_RANGE)
cg = np.polyfit(n_arr, np.log(np.array(results["Global cost"]) + 1e-30), 1)
cp = np.polyfit(n_arr, np.log(np.array(results["PDE-constrained"]) + 1e-30), 1)

ax.set_xlabel("Number of qubits", fontsize=13)
ax.set_ylabel("Gradient variance", fontsize=13)
ax.set_title("H4: Gradient Variance Decay by Cost Type", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.text(0.95, 0.95,
        f"Global decay: exp({cg[0]:.2f}n)\nPDE decay: exp({cp[0]:.2f}n)",
        transform=ax.transAxes, fontsize=11, va="top", ha="right",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
os.makedirs('results/H4', exist_ok=True)
plt.savefig('results/H4/plot.png', dpi=120)
plt.show()
print(f"Hypothesis supported: {cp[0] > cg[0]} (PDE decay less negative)")